In [2]:
"""
Silhouette Heatmap — Entropy + Active Window Rate Combined
===========================================================
Paper: Beyond Code: Characterizing Documentation Governance
       and Coordination in Open Source Projects

PURPOSE
-------
Displays silhouette scores for the combined Entropy +
Active Window Rate feature set across k = 2 to k = 6.
Justifies the selection of k=3 as the number of archetypes.

Only the combined feature set is shown because the
methodology section already establishes the theoretical
and empirical rationale for combining both metrics.
The individual metric rows are intentionally excluded
to keep the figure focused on a single argument:
k=3 is the peak for this combination.

OUTPUTS
-------
  silhouette_heatmap_combined_v2.pdf
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
DATA_PATH   = "combined_doc_done.csv"   # update path if needed
OUTPUT_PATH = "silhouette_heatmap_combined_v2.pdf"
FEATURES    = ['entropy_norm', 'active_window_rate']
K_RANGE     = list(range(2, 7))         # k = 2, 3, 4, 5, 6
SELECTED_K  = 3
RANDOM_SEED = 42

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} repositories.\n")

# ─────────────────────────────────────────────
# COMPUTE SILHOUETTE SCORES
# ─────────────────────────────────────────────
scaler = StandardScaler()
X      = scaler.fit_transform(df[FEATURES])

scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    km.fit(X)
    scores.append(round(silhouette_score(X, km.labels_), 4))

print("Entropy + ActiveWin silhouette scores:")
for k, s in zip(K_RANGE, scores):
    marker = "  ← selected" if k == SELECTED_K else ""
    print(f"  k={k}  {s:.4f}{marker}")

sel_col = K_RANGE.index(SELECTED_K)
best_k  = K_RANGE[scores.index(max(scores))]

# ─────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 2.6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

matrix = np.array([scores])

# Blues colormap — professional, clean, print-friendly
im = ax.imshow(matrix, cmap='Blues', aspect='auto',
               vmin=min(scores) - 0.02,
               vmax=max(scores) + 0.02)

# ── Axis labels ──
ax.set_xticks(range(len(K_RANGE)))
ax.set_xticklabels([f'k = {k}' for k in K_RANGE],
                   color='#1a1a1a', fontsize=13, fontweight='bold')
ax.set_yticks([0])
ax.set_yticklabels(['Entropy + Active\nWindow Rate'],
                   color='#1a1a1a', fontsize=11, fontweight='bold')
ax.tick_params(colors='#1a1a1a', length=0, pad=12)
for spine in ax.spines.values():
    spine.set_edgecolor('#DDDDDD')

# ── Annotate cells ──
for j, score in enumerate(scores):
    is_selected = (j == sel_col)
    is_best     = (K_RANGE[j] == best_k)
    symbol      = ' ★' if is_best else ''
    weight      = 'bold' if is_selected else 'normal'
    fontsize    = 15 if is_selected else 12
    # White text on darker cells, dark text on lighter cells
    text_color  = 'white' if score >= 0.58 else '#1a1a1a'

    ax.text(j, 0, f'{score:.3f}{symbol}',
            ha='center', va='center',
            color=text_color,
            fontsize=fontsize,
            fontweight=weight)

# ── Selected cell border (k=3) ──
ax.add_patch(plt.Rectangle(
    (sel_col - 0.5, -0.5), 1, 1,
    linewidth=4, edgecolor='#1F4E79',
    facecolor='none', zorder=6
))

# ── Colorbar ──
cbar = plt.colorbar(im, ax=ax, pad=0.015,
                    shrink=0.88, aspect=10)
cbar.ax.yaxis.set_tick_params(color='#333333', labelsize=9)
cbar.ax.set_ylabel('Silhouette Score',
                   color='#333333', fontsize=9)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#333333')
cbar.outline.set_edgecolor('#CCCCCC')

# ── Legend ──
legend_elements = [
    mpatches.Patch(facecolor='#1F4E79', edgecolor='#1F4E79',
                   label=f'Selected: k=3  '
                         f'(silhouette = {scores[sel_col]:.3f})'),
    plt.Line2D([0], [0], marker='*', color='#1a1a1a',
               linestyle='None', markersize=12,
               label='★  peak silhouette'),
]
ax.legend(handles=legend_elements,
          loc='lower left',
          bbox_to_anchor=(0.0, -0.62),
          facecolor='white', labelcolor='#1a1a1a',
          framealpha=0.95, fontsize=10.5, ncol=2,
          edgecolor='#CCCCCC')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight',
            facecolor='white', format='pdf')
plt.close()
print(f"\nSaved: {OUTPUT_PATH}")

Loaded 100 repositories.

Entropy + ActiveWin silhouette scores:
  k=2  0.5471
  k=3  0.6032  ← selected
  k=4  0.5912
  k=5  0.5545
  k=6  0.5178

Saved: silhouette_heatmap_combined_v2.pdf


In [1]:
"""
Archetype Clustering & Descriptive Statistics
==============================================
Paper: Beyond Code: Characterizing Documentation Governance
       and Coordination in Open Source Projects

PURPOSE
-------
1. Assigns each repository to one of three rhythm archetypes
   (Consistent, Occasional, Sparse) using K-Means clustering
   on entropy_norm + active_window_rate.
2. Computes mean, median, SD, min, max for each archetype
   across both metrics — matching Table 3 in the paper.
3. Exports labeled repositories to CSV for inspection.

OUTPUTS
-------
  archetype_assignments.csv   — all repos with archetype label
  archetype_stats.csv         — Table 3 descriptive statistics
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION — update path as needed
# ─────────────────────────────────────────────
DATA_PATH        = "combined_doc_done.csv"
OUTPUT_ASSIGNED  = "archetype_assignments.csv"
OUTPUT_STATS     = "archetype_stats.csv"
FEATURES         = ['entropy_norm', 'active_window_rate']
SELECTED_K       = 3
RANDOM_SEED      = 42

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} repositories.")
print(f"Columns: {list(df.columns)}\n")

# ─────────────────────────────────────────────
# CLUSTER
# ─────────────────────────────────────────────
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

km = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_SEED, n_init=10)
km.fit(X)
df['cluster_id'] = km.labels_

sil = silhouette_score(X, km.labels_)
print(f"Silhouette score (k={SELECTED_K}): {sil:.4f}\n")

# ─────────────────────────────────────────────
# NAME ARCHETYPES by entropy rank
# Consistent  = highest mean entropy_norm
# Occasional  = middle
# Sparse      = lowest mean entropy_norm
# ─────────────────────────────────────────────
cluster_means = df.groupby('cluster_id')['entropy_norm'].mean().sort_values(ascending=False)
rank_to_name  = {
    cluster_means.index[0]: 'Consistent',
    cluster_means.index[1]: 'Occasional',
    cluster_means.index[2]: 'Sparse',
}
df['archetype'] = df['cluster_id'].map(rank_to_name)

# ─────────────────────────────────────────────
# COUNTS
# ─────────────────────────────────────────────
counts = df['archetype'].value_counts().reindex(
    ['Consistent', 'Occasional', 'Sparse'])
print("Archetype counts:")
for name, count in counts.items():
    pct = count / len(df) * 100
    print(f"  {name:12s}: n={count}  ({pct:.0f}%)")
print()

# ─────────────────────────────────────────────
# DESCRIPTIVE STATISTICS — Table 3
# ─────────────────────────────────────────────
rows = []
for archetype in ['Consistent', 'Occasional', 'Sparse']:
    subset = df[df['archetype'] == archetype]
    n = len(subset)
    for metric, label in [('entropy_norm', 'H_norm'),
                           ('active_window_rate', 'AWR')]:
        vals = subset[metric]
        rows.append({
            'Archetype' : archetype,
            'n'         : n,
            'Metric'    : label,
            'Mean'      : round(vals.mean(), 3),
            'Median'    : round(vals.median(), 3),
            'SD'        : round(vals.std(), 3),
            'Min'       : round(vals.min(), 3),
            'Max'       : round(vals.max(), 3),
        })

stats_df = pd.DataFrame(rows)

print("Table 3 — Descriptive statistics per archetype:")
print(stats_df.to_string(index=False))
print()

# ─────────────────────────────────────────────
# SAVE OUTPUTS
# ─────────────────────────────────────────────
df.to_csv(OUTPUT_ASSIGNED, index=False)
stats_df.to_csv(OUTPUT_STATS, index=False)

print(f"Saved: {OUTPUT_ASSIGNED}")
print(f"Saved: {OUTPUT_STATS}")
print()

# ─────────────────────────────────────────────
# QUICK SANITY CHECK — match paper numbers
# ─────────────────────────────────────────────
print("Sanity check against paper (Table 3):")
expected = {
    ('Consistent', 'H_norm') : (0.904, 0.900, 0.052),
    ('Consistent', 'AWR')    : (0.854, 0.861, 0.119),
    ('Occasional', 'H_norm') : (0.719, 0.722, 0.067),
    ('Occasional', 'AWR')    : (0.391, 0.393, 0.094),
    ('Sparse',     'H_norm') : (0.402, 0.460, 0.138),
    ('Sparse',     'AWR')    : (0.125, 0.131, 0.070),
}
all_match = True
for (arch, metric), (exp_mean, exp_med, exp_sd) in expected.items():
    row = stats_df[(stats_df['Archetype'] == arch) &
                   (stats_df['Metric'] == metric)].iloc[0]
    match = (abs(row['Mean']   - exp_mean) < 0.005 and
             abs(row['Median'] - exp_med)  < 0.005 and
             abs(row['SD']     - exp_sd)   < 0.005)
    status = "✓" if match else "✗ MISMATCH"
    print(f"  {arch:12s} {metric:7s}: mean={row['Mean']:.3f} "
          f"med={row['Median']:.3f} sd={row['SD']:.3f}  {status}")
    if not match:
        all_match = False

print()
if all_match:
    print("All values match paper. Archetype assignments validated.")
else:
    print("Some values differ — check clustering or data pipeline.")

Loaded 100 repositories.
Columns: ['project_name', 'repo', 'active_window_rate', 'entropy_norm']

Silhouette score (k=3): 0.6032

Archetype counts:
  Consistent  : n=32  (32%)
  Occasional  : n=51  (51%)
  Sparse      : n=17  (17%)

Table 3 — Descriptive statistics per archetype:
 Archetype  n Metric  Mean  Median    SD   Min   Max
Consistent 32 H_norm 0.904   0.900 0.052 0.769 0.989
Consistent 32    AWR 0.854   0.861 0.119 0.672 1.000
Occasional 51 H_norm 0.719   0.722 0.067 0.563 0.830
Occasional 51    AWR 0.391   0.393 0.094 0.213 0.574
    Sparse 17 H_norm 0.402   0.460 0.138 0.137 0.570
    Sparse 17    AWR 0.117   0.115 0.060 0.033 0.230

Saved: archetype_assignments.csv
Saved: archetype_stats.csv

Sanity check against paper (Table 3):
  Consistent   H_norm : mean=0.904 med=0.900 sd=0.052  ✓
  Consistent   AWR    : mean=0.854 med=0.861 sd=0.119  ✓
  Occasional   H_norm : mean=0.719 med=0.722 sd=0.067  ✓
  Occasional   AWR    : mean=0.391 med=0.393 sd=0.094  ✓
  Sparse       H_nor

In [2]:
"""
Contributor Count by Archetype Analysis
========================================
Paper: Beyond Code: Characterizing Documentation Governance
       and Coordination in Open Source Projects

PURPOSE
-------
Answers: Do Sparse Documenter projects have fewer contributors
than Consistent and Occasional Documenters?

Merges archetype assignments (from entropy + AWR clustering)
with total contributor counts per repository, then computes
descriptive statistics and significance tests per archetype.

INPUTS
------
  combined_doc_done.csv        — must have: repo, entropy_norm,
                                 active_window_rate
  combined_contributors.csv    — must have: repo,
                                 unique_contributors_for_metrics

OUTPUTS
-------
  archetype_contributor_stats.csv   — descriptive stats per archetype
  archetype_contributor_full.csv    — per-repo with archetype label
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION — update paths if needed
# ─────────────────────────────────────────────
DOC_PATH      = "combined_doc_done.csv"
CONTRIB_PATH  = "combined_contributors.csv"
OUT_STATS     = "archetype_contributor_stats.csv"
OUT_FULL      = "archetype_contributor_full.csv"

FEATURES      = ['entropy_norm', 'active_window_rate']
SELECTED_K    = 3
RANDOM_SEED   = 42

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────
doc     = pd.read_csv(DOC_PATH)
contrib = pd.read_csv(CONTRIB_PATH)

print(f"Doc file    : {len(doc)} repos")
print(f"Contrib file: {len(contrib)} repos")
print()

# ─────────────────────────────────────────────
# ASSIGN ARCHETYPES
# ─────────────────────────────────────────────
scaler = StandardScaler()
X = scaler.fit_transform(doc[FEATURES])

km = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_SEED, n_init=10)
km.fit(X)
doc['cluster_id'] = km.labels_

sil = silhouette_score(X, km.labels_)
print(f"Silhouette score (k={SELECTED_K}): {sil:.4f}")

# Name by entropy rank: highest = Consistent, middle = Occasional,
# lowest = Sparse
cluster_means = (doc.groupby('cluster_id')['entropy_norm']
                   .mean()
                   .sort_values(ascending=False))
rank_to_name = {
    cluster_means.index[0]: 'Consistent',
    cluster_means.index[1]: 'Occasional',
    cluster_means.index[2]: 'Sparse',
}
doc['archetype'] = doc['cluster_id'].map(rank_to_name)

print("\nArchetype counts:")
counts = doc['archetype'].value_counts().reindex(
    ['Consistent', 'Occasional', 'Sparse'])
for name, n in counts.items():
    print(f"  {name:12s}: n={n} ({n/len(doc)*100:.0f}%)")

# ─────────────────────────────────────────────
# MERGE
# ─────────────────────────────────────────────
merged = doc.merge(
    contrib[['repo', 'unique_contributors_for_metrics']],
    on='repo', how='left'
)

missing = merged['unique_contributors_for_metrics'].isna().sum()
if missing > 0:
    print(f"\nWARNING: {missing} repos missing contributor data after merge.")

# ─────────────────────────────────────────────
# DESCRIPTIVE STATISTICS PER ARCHETYPE
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("CONTRIBUTOR COUNT BY ARCHETYPE")
print("(unique_contributors_for_metrics = C_all after bot filtering)")
print("="*60)

rows = []
for arch in ['Consistent', 'Occasional', 'Sparse']:
    vals = merged[merged['archetype'] == arch][
        'unique_contributors_for_metrics'].dropna()
    row = {
        'Archetype' : arch,
        'n'         : len(vals),
        'Mean'      : round(vals.mean(), 1),
        'Median'    : round(vals.median(), 1),
        'SD'        : round(vals.std(), 1),
        'Min'       : int(vals.min()),
        'Max'       : int(vals.max()),
    }
    rows.append(row)
    print(f"\n{arch} (n={row['n']}):")
    print(f"  Mean   : {row['Mean']}")
    print(f"  Median : {row['Median']}")
    print(f"  SD     : {row['SD']}")
    print(f"  Min    : {row['Min']}")
    print(f"  Max    : {row['Max']}")

stats_df = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# SIGNIFICANCE TESTS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("SIGNIFICANCE TESTS")
print("="*60)

c_vals = merged[merged['archetype']=='Consistent'][
    'unique_contributors_for_metrics'].dropna()
o_vals = merged[merged['archetype']=='Occasional'][
    'unique_contributors_for_metrics'].dropna()
s_vals = merged[merged['archetype']=='Sparse'][
    'unique_contributors_for_metrics'].dropna()

# Kruskal-Wallis (non-parametric, appropriate given skewed distribution)
h, p = stats.kruskal(c_vals, o_vals, s_vals)
print(f"\nKruskal-Wallis: H={h:.2f}, p={p:.4f}")
if p < 0.001:
    print("  → Highly significant difference across archetypes (p<0.001)")

# Pairwise Mann-Whitney
print("\nMann-Whitney pairwise (two-sided):")
pairs = [
    ('Consistent', c_vals, 'Sparse',     s_vals),
    ('Occasional', o_vals, 'Sparse',     s_vals),
    ('Consistent', c_vals, 'Occasional', o_vals),
]
for a_name, a_data, b_name, b_data in pairs:
    u, p_val = stats.mannwhitneyu(a_data, b_data, alternative='two-sided')
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else \
          "*" if p_val < 0.05 else "ns"
    print(f"  {a_name:12s} vs {b_name:12s}: "
          f"U={u:.0f}, p={p_val:.4f}  {sig}")

# ─────────────────────────────────────────────
# PAPER-READY SENTENCE
# ─────────────────────────────────────────────
c_med = stats_df[stats_df['Archetype']=='Consistent']['Median'].values[0]
o_med = stats_df[stats_df['Archetype']=='Occasional']['Median'].values[0]
s_med = stats_df[stats_df['Archetype']=='Sparse']['Median'].values[0]

print("\n" + "="*60)
print("PAPER-READY SENTENCE:")
print("="*60)
print(f"""
Consistent with this interpretation, Sparse Documenters have
a median of only {s_med:.0f} total contributors compared to
{o_med:.0f} for Occasional and {c_med:.0f} for Consistent
Documenters (Kruskal-Wallis H={h:.2f}, p<0.0001), confirming
that sparse rhythm co-occurs with substantially thinner
contributor ecosystems.
""")

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
stats_df.to_csv(OUT_STATS, index=False)
merged.to_csv(OUT_FULL, index=False)
print(f"Saved: {OUT_STATS}")
print(f"Saved: {OUT_FULL}")

Doc file    : 100 repos
Contrib file: 100 repos

Silhouette score (k=3): 0.6032

Archetype counts:
  Consistent  : n=32 (32%)
  Occasional  : n=51 (51%)
  Sparse      : n=17 (17%)

CONTRIBUTOR COUNT BY ARCHETYPE
(unique_contributors_for_metrics = C_all after bot filtering)

Consistent (n=32):
  Mean   : 1105.7
  Median : 598.0
  SD     : 1649.5
  Min    : 58
  Max    : 8008

Occasional (n=51):
  Mean   : 532.1
  Median : 318.0
  SD     : 625.3
  Min    : 9
  Max    : 3456

Sparse (n=17):
  Mean   : 102.4
  Median : 16.0
  SD     : 161.2
  Min    : 1
  Max    : 578

SIGNIFICANCE TESTS

Kruskal-Wallis: H=26.56, p=0.0000
  → Highly significant difference across archetypes (p<0.001)

Mann-Whitney pairwise (two-sided):
  Consistent   vs Sparse      : U=495, p=0.0000  ***
  Occasional   vs Sparse      : U=734, p=0.0000  ***
  Consistent   vs Occasional  : U=1035, p=0.0409  *

PAPER-READY SENTENCE:

Consistent with this interpretation, Sparse Documenters have
a median of only 16 total contrib